# ⬡ SkillChain — Explorer la blockchain

Ce notebook est votre fil rouge pour le workshop.
Chaque cellule explore un concept différent de la blockchain **en lisant des données réelles** sur le réseau Sepolia.

---

## 🚀 Avant de commencer — 2 étapes

**1. Faites une copie personnelle**  
Menu **Fichier → Enregistrer une copie dans Drive**  
→ Vous travaillez sur votre propre copie. L'original n'est jamais modifié.

**2. Exécutez les cellules dans l'ordre**  
Cliquer ▶ à gauche de chaque cellule, ou **Shift+Entrée**  
→ Les cellules avec un 🔧 ont un **champ de saisie** — remplissez-le, puis exécutez.

---
> **Réseau :** Sepolia Testnet — clone d'Ethereum, ETH sans valeur réelle  
> **Contrat SkillBadge :** déployé et actif sur ce réseau

## 0. Installation

In [ ]:
!pip install web3 -q
print("✓ web3.py installé")

## 1. Connexion au réseau Sepolia

On se connecte via un nœud RPC public — aucune clé API requise.  
La lecture de la blockchain est **publique et gratuite**.

In [ ]:
from web3 import Web3
from datetime import datetime, timezone

RPC_URL = "https://ethereum-sepolia.publicnode.com"
w3 = Web3(Web3.HTTPProvider(RPC_URL))

if w3.is_connected():
    print(f"✓ Connecté à Sepolia")
    print(f"  Dernier bloc : #{w3.eth.block_number:,}  (augmente toutes les ~12 sec)")
else:
    print("✗ Connexion échouée — vérifiez votre connexion internet")

## 2. 🔧 Votre contrat SkillBadge

Entrez l'adresse de votre contrat déployé au Bloc 3.  
Elle est visible dans Remix (panneau Deployed Contracts) ou sur Etherscan.

In [ ]:
CONTRACT_ADDRESS = "0x679E528c0712cA4F1Dbe37Ab99Dfd66855091fa8" #@param {type:"string"}

ABI = [
    {
        "inputs": [],
        "name": "getBadgeCount",
        "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [{"internalType": "address", "name": "student", "type": "address"}],
        "name": "getBadgesOf",
        "outputs": [{"internalType": "uint256[]", "name": "", "type": "uint256[]"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [{"internalType": "uint256", "name": "index", "type": "uint256"}],
        "name": "getBadge",
        "outputs": [
            {"internalType": "address", "name": "", "type": "address"},
            {"internalType": "string",  "name": "", "type": "string"},
            {"internalType": "uint256", "name": "", "type": "uint256"},
            {"internalType": "address", "name": "", "type": "address"}
        ],
        "stateMutability": "view",
        "type": "function"
    }
]

address = Web3.to_checksum_address(CONTRACT_ADDRESS)
contract = w3.eth.contract(address=address, abi=ABI)

print(f"✓ Contrat chargé : {address}")
print(f"  Etherscan : https://sepolia.etherscan.io/address/{address}")

## 3. Combien de badges sur votre contrat ?

`getBadgeCount()` est une fonction `view` — lecture seule, pas de gas.

In [ ]:
count = contract.functions.getBadgeCount().call()
print(f"Badges sur ce contrat : {count}")
if count == 0:
    print("  → Émettez un badge depuis Remix ou la DApp, puis relancez cette cellule.")

## 4. Lire tous les badges

Ces données sont **stockées sur la blockchain** — permanentes, non modifiables.  
`block.timestamp` : horodatage fourni par le validateur du bloc.

In [ ]:
def fmt_addr(addr):
    return f"{addr[:6]}...{addr[-4:]}"

def fmt_time(ts):
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%d/%m/%Y %H:%M UTC")

count = contract.functions.getBadgeCount().call()

if count == 0:
    print("Aucun badge. Émettez-en un puis relancez.")
else:
    print(f"{'#':<4} {'Compétence':<28} {'Étudiant':<14} {'Date':<22} {'Émetteur':<14}")
    print("─" * 90)
    for i in range(count):
        student, skill, issued_at, issuer = contract.functions.getBadge(i).call()
        print(f"{i:<4} {skill:<28} {fmt_addr(student):<14} {fmt_time(issued_at):<22} {fmt_addr(issuer):<14}")

## 5. 🔧 Vos badges reçus

Entrez votre adresse MetaMask pour voir les badges qui vous ont été attribués.

In [ ]:
MON_ADRESSE = "0x" #@param {type:"string"}

if MON_ADRESSE == "0x" or len(MON_ADRESSE) != 42:
    print("⚠️  Entrez votre adresse MetaMask complète (42 caractères, commence par 0x)")
else:
    checksum = Web3.to_checksum_address(MON_ADRESSE)
    indexes  = contract.functions.getBadgesOf(checksum).call()
    print(f"Adresse : {checksum}")
    print(f"Badges  : {len(indexes)}")
    print()
    if len(indexes) == 0:
        print("  Aucun badge. Demandez à un voisin d'en émettre un vers votre adresse.")
    else:
        for idx in indexes:
            student, skill, issued_at, issuer = contract.functions.getBadge(idx).call()
            print(f"  Badge #{idx}")
            print(f"    Compétence : {skill}")
            print(f"    Émis le    : {fmt_time(issued_at)}")
            print(f"    Par        : {issuer}")
            print()

## 6. Explorer un bloc

Un bloc = transactions validées à un instant donné.  
Chaque bloc référence le hash du précédent → c'est la **chaîne**.

In [ ]:
bloc = w3.eth.get_block('latest')

print(f"Bloc #{bloc['number']:,}")
print(f"  Timestamp      : {fmt_time(bloc['timestamp'])}")
print(f"  Hash           : {bloc['hash'].hex()}")
print(f"  Hash précédent : {bloc['parentHash'].hex()}")
print(f"  Transactions   : {len(bloc['transactions'])}")
print(f"  Gas utilisé    : {bloc['gasUsed']:,}")
print()
print("→ 'parentHash' = le hash du bloc précédent.")
print("  Modifier un ancien bloc changerait son hash → casserait ce lien → invaliderait tous les suivants.")

## 7. 🔧 Explorer une transaction

Entrez le hash d'une de vos transactions `issueBadge`.  
Récupérez-le depuis Etherscan ou depuis le lien généré par la DApp.

In [ ]:
TX_HASH = "0x" #@param {type:"string"}

if TX_HASH == "0x" or len(TX_HASH) < 60:
    print("⚠️  Entrez un hash de transaction valide (66 caractères, commence par 0x)")
else:
    tx      = w3.eth.get_transaction(TX_HASH)
    receipt = w3.eth.get_transaction_receipt(TX_HASH)
    bloc_tx = w3.eth.get_block(tx['blockNumber'])
    statut  = "✓ Succès" if receipt['status'] == 1 else "✗ Échec"

    print(f"Statut       : {statut}")
    print(f"Bloc         : #{tx['blockNumber']:,}")
    print(f"Timestamp    : {fmt_time(bloc_tx['timestamp'])}")
    print()
    print(f"De           : {tx['from']}  ← msg.sender dans le contrat")
    print(f"Vers contrat : {tx['to']}")
    print()
    gas_price_gwei = tx['gasPrice'] / 1e9
    cout_eth = (receipt['gasUsed'] * tx['gasPrice']) / 1e18
    print(f"Gas utilisé  : {receipt['gasUsed']:,} unités")
    print(f"Gas price    : {gas_price_gwei:.4f} Gwei")
    print(f"Coût total   : {cout_eth:.8f} ETH")
    print(f"Mainnet eq.  : ~{receipt['gasUsed'] * 20 / 1e9:.6f} ETH à 20 Gwei ≈ 3-5€")

## 8. Le hash — démonstration

Ethereum utilise **keccak256**. Trois propriétés fondamentales :

In [ ]:
from eth_hash.auto import keccak

def show_hash(label, text):
    h = keccak(text.encode('utf-8')).hex()
    print(f"  {label:<32} →  0x{h}")

print("1. Déterministe — même entrée = même sortie")
show_hash('"Bonjour" (1ère fois)', "Bonjour")
show_hash('"Bonjour" (2ème fois)', "Bonjour")
print()

print("2. Effet avalanche — 1 caractère change = hash méconnaissable")
show_hash('"Bonjour" (B majuscule)', "Bonjour")
show_hash('"bonjour" (b minuscule)', "bonjour")
print()

print("3. Taille fixe — toujours 32 octets peu importe l'entrée")
show_hash('1 caractère', "A")
show_hash('"Solidity Basics"', "Solidity Basics")
show_hash('"A" × 10 000', "A" * 10000)

## 9. Dérivation d'une adresse Ethereum

Une adresse n'est pas aléatoire — elle est **dérivée de votre clé privée**.  
Démonstration avec une clé de test (jamais entrer une vraie clé ici).

In [ ]:
from eth_account import Account

CLE_DEMO = "0xaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa"
compte = Account.from_key(CLE_DEMO)

print(f"Clé privée : {CLE_DEMO}")
print(f"Adresse    : {compte.address}")
print()
print("La même clé donne toujours la même adresse :")
print(f"  Identique ? {Account.from_key(CLE_DEMO).address == compte.address}")
print()
print("→ MetaMask stocke votre clé privée chiffrée localement.")
print("  La seed phrase (12 mots) permet de la régénérer — ne la partagez jamais.")

## 10. État du réseau Sepolia

In [ ]:
dernier = w3.eth.get_block('latest')
precedent = w3.eth.get_block(dernier['number'] - 1)

print("Réseau Sepolia — en ce moment")
print(f"  Bloc actuel           : #{dernier['number']:,}")
print(f"  Temps entre blocs     : {dernier['timestamp'] - precedent['timestamp']}s  (cible : 12s sur PoS)")
print(f"  Transactions ce bloc  : {len(dernier['transactions'])}")
print(f"  Base fee              : {dernier['baseFeePerGas'] / 1e9:.4f} Gwei")
print()
print(f"  Votre contrat est dans l'un de ces {dernier['number']:,} blocs. Il y restera pour toujours.")

---

## Récapitulatif

| Cellule | Concept |
|---|---|
| 1 | Connexion à Sepolia via RPC |
| 2 🔧 | Charger son propre contrat |
| 3–5 🔧 | Lire ses badges on-chain |
| 6 | Structure d'un bloc et chaînage |
| 7 🔧 | Anatomie d'une transaction |
| 8 | Hash : déterminisme, avalanche, taille fixe |
| 9 | Dérivation d'adresse depuis clé privée |
| 10 | État du réseau en temps réel |

---
> **Prochaine étape :** DApp SkillChain pour émettre des badges (nécessite MetaMask).